# 01 Loading Data

Load small CSV files into Spark DataFrames and inspect schemas before writing SQL.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("module-02-spark-sql").getOrCreate()
base = "../../datasets/module_02"

In [3]:
municipalities = spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/municipalities.csv")
accessibility = spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/accessibility_scores.csv")
poi = spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/poi_counts.csv")
property_values = spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/property_values.csv")
municipalities.printSchema()
municipalities.show(5, truncate=False)

root
 |-- municipality_id: integer (nullable = true)
 |-- municipality_name: string (nullable = true)
 |-- canton: string (nullable = true)
 |-- population: integer (nullable = true)

+---------------+-----------------+------+----------+
|municipality_id|municipality_name|canton|population|
+---------------+-----------------+------+----------+
|1              |Zurich           |ZH    |421878    |
|2              |Winterthur       |ZH    |116122    |
|3              |Uster            |ZH    |36012     |
|4              |Meilen           |ZH    |14733     |
|5              |Bern             |BE    |134591    |
+---------------+-----------------+------+----------+
only showing top 5 rows



In [4]:
accessibility.printSchema()

root
 |-- municipality_id: integer (nullable = true)
 |-- accessibility_score: integer (nullable = true)



In [5]:
for name, df in [("municipalities", municipalities), ("accessibility", accessibility), ("poi", poi), ("property_values", property_values)]:
    print(name, df.count())

municipalities 30
accessibility 30
poi 30
property_values 30


In [25]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

municipalities_schema = StructType([
    StructField("municipality_id", IntegerType(), True),
    StructField("municipality_name", StringType(), True),
    StructField("canton", StringType(), True),
    StructField("population", IntegerType(), True),
])

municipalities = (
    spark.read
    .option("header", True)
    .schema(municipalities_schema)
    .csv(f"{base}/municipalities.csv")
)

ConnectionRefusedError: [Errno 111] Connection refused

In [9]:
municipalities.printSchema()
municipalities.show(5)
municipalities.count()

root
 |-- municipality_id: integer (nullable = true)
 |-- municipality_name: string (nullable = true)
 |-- canton: string (nullable = true)
 |-- population: integer (nullable = true)

+---------------+-----------------+------+----------+
|municipality_id|municipality_name|canton|population|
+---------------+-----------------+------+----------+
|              1|           Zurich|    ZH|    421878|
|              2|       Winterthur|    ZH|    116122|
|              3|            Uster|    ZH|     36012|
|              4|           Meilen|    ZH|     14733|
|              5|             Bern|    BE|    134591|
+---------------+-----------------+------+----------+
only showing top 5 rows



30

In [10]:
municipalities.createOrReplaceTempView("municipalities")

In [11]:
spark.sql("""
SELECT canton, COUNT(*) AS municipality_count
FROM municipalities
GROUP BY canton
ORDER BY municipality_count DESC
""").show()

+------+------------------+
|canton|municipality_count|
+------+------------------+
|    ZH|                 4|
|    BE|                 4|
|    VD|                 3|
|    TI|                 3|
|    GR|                 3|
|    SG|                 3|
|    GE|                 3|
|    LU|                 3|
|    BS|                 2|
|    NE|                 2|
+------+------------------+



In [22]:
accessibility_score_schema = StructType([
    StructField("municipality_id", IntegerType(), True),
    StructField("accessibility_score", DoubleType(), True),
])
spark.conf.set("spark.sql.shuffle.partitions", "4")

accessibility_scores = (
    spark.read
    .option("header", True)
    .schema(accessibility_score_schema)
    .csv(f"{base}/accessibility_scores.csv")
)
accessibility_scores.createOrReplaceTempView("accessibility_scores")

NameError: name 'DoubleType' is not defined

In [18]:
spark.sql("""
SELECT
  m.municipality_id,
  m.municipality_name,
  m.canton,
  m.population,
  a.accessibility_score
FROM municipalities m
JOIN accessibility_scores a
  ON m.municipality_id = a.municipality_id
  ORDER BY a.municipality_id
  """).show()

+---------------+-----------------+------+----------+-------------------+
|municipality_id|municipality_name|canton|population|accessibility_score|
+---------------+-----------------+------+----------+-------------------+
|              1|           Zurich|    ZH|    421878|                 97|
|              2|       Winterthur|    ZH|    116122|                 95|
|              3|            Uster|    ZH|     36012|                 92|
|              4|           Meilen|    ZH|     14733|                 91|
|              5|             Bern|    BE|    134591|                 98|
|              6|      Biel/Bienne|    BE|     55206|                 98|
|              7|             Thun|    BE|     43850|                 98|
|              8|       Interlaken|    BE|      5621|                 82|
|              9|            Basel|    BS|    173552|                 96|
|             10|           Riehen|    BS|     21339|                 91|
|             11|           Geneva|   

## Query plan

In [20]:
query = spark.sql("""
SELECT
  canton,
  COUNT(*) AS municipality_count
FROM municipalities
GROUP BY canton
""")
query.show()

query.explain("formatted")

+------+------------------+
|canton|municipality_count|
+------+------------------+
|    BS|                 2|
|    VD|                 3|
|    TI|                 3|
|    NE|                 2|
|    ZH|                 4|
|    GR|                 3|
|    SG|                 3|
|    GE|                 3|
|    BE|                 4|
|    LU|                 3|
+------+------------------+

== Physical Plan ==
AdaptiveSparkPlan (5)
+- HashAggregate (4)
   +- Exchange (3)
      +- HashAggregate (2)
         +- Scan csv  (1)


(1) Scan csv 
Output [1]: [canton#142]
Batched: false
Location: InMemoryFileIndex [file:/home/jovyan/work/datasets/module_02/municipalities.csv]
ReadSchema: struct<canton:string>

(2) HashAggregate
Input [1]: [canton#142]
Keys [1]: [canton#142]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#353L]
Results [2]: [canton#142, count#354L]

(3) Exchange
Input [2]: [canton#142, count#354L]
Arguments: hashpartitioning(canton#142, 200), ENSURE_REQUIREMENT

In [21]:
query = spark.sql("""
SELECT
  m.canton,
  COUNT(*) AS municipalities,
  AVG(a.accessibility_score) AS avg_accessibility
FROM municipalities m
JOIN accessibility_scores a
  ON m.municipality_id = a.municipality_id
GROUP BY m.canton
ORDER BY avg_accessibility DESC
""")
query.show()

query.explain("formatted")

+------+--------------+-----------------+
|canton|municipalities|avg_accessibility|
+------+--------------+-----------------+
|    GE|             3|97.33333333333333|
|    LU|             3|             95.0|
|    BE|             4|             94.0|
|    ZH|             4|            93.75|
|    BS|             2|             93.5|
|    SG|             3|92.66666666666667|
|    VD|             3|91.66666666666667|
|    TI|             3|91.66666666666667|
|    NE|             2|             90.0|
|    GR|             3|             83.0|
+------+--------------+-----------------+

== Physical Plan ==
AdaptiveSparkPlan (13)
+- Sort (12)
   +- Exchange (11)
      +- HashAggregate (10)
         +- Exchange (9)
            +- HashAggregate (8)
               +- Project (7)
                  +- BroadcastHashJoin Inner BuildRight (6)
                     :- Filter (2)
                     :  +- Scan csv  (1)
                     +- BroadcastExchange (5)
                        +- Filter (4)

Reflection: Which part of this notebook defines data location, and which part asks Spark to execute work?